# 제17장: 금융 AI 거버넌스

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/back2zion/ai-governance-handbook/blob/main/notebooks/ch17.ipynb)

> **『AI 거버넌스 실무 지침서』** (곽두일) 실습 코드
>
> 이 노트북의 모든 코드는 Google Colab에서 바로 실행할 수 있습니다.

금융 AI는 SR 11-7(모델 리스크 관리), Basel III, GDPR이 교차하는 복합 규제 환경입니다.
**SHAP 기반 신용 공정성 감사, SR 11-7 모델 검증, AML 이상거래 탐지,
스트레스 테스트, 알고리즘 트레이딩 거버넌스** 구현을 실습합니다.

In [ ]:
# 필요 패키지 설치 (최초 1회)
!pip install -q fairlearn shap mlflow diffprivlib alibi alibi-detect evidently torch

**Compliance with Article 36-2: SHAP-based Automated Assessment Explanation System**

In [ ]:
import shap
import numpy as np
import pandas as pd
from dataclasses import dataclass
from datetime import datetime

@dataclass
class CreditExplanation:
 customer_id: str; score: int; decision: str
 positive_factors: list; negative_factors: list
 counterfactual: str = ""

class CreditExplanationService:
 """Compliance service for Article 36-2 of the Credit Information Act"""
 def __init__(self, model, feature_names):
 self.explainer = shap.TreeExplainer(model)
 self.feature_names = feature_names

 def explain_to_customer(self, customer_id, x_input, score, decision):
 # Calculate SHAP values
 shap_values = self.explainer.shap_values(x_input)

 # Extract top features (converted to customer-friendly terms)
 factors = sorted(zip(self.feature_names, shap_values), key=lambda x: abs(x[1]), reverse=True)
 pos = [f[0] for f in factors if f[1] > 0][:3]
 neg = [f[0] for f in factors if f[1] < 0][:3]

 return CreditExplanation(
 customer_id=customer_id, score=score, decision=decision,
 positive_factors=pos, negative_factors=neg,
 counterfactual="Reducing your debt-to-income ratio by 5% will increase approval probability."
 )

**SR 11-7 Model Validation Three-Stage Framework Implementation**

In [ ]:
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
from datetime import datetime
from sklearn.metrics import roc_auc_score, brier_score_loss
from scipy import stats

@dataclass
class ValidationResult:
    stage: str
    test_name: str
    passed: bool
    score: float
    threshold: float
    details: str
    timestamp: datetime = field(default_factory=datetime.now)

> 아래 셀을 실행하여 결과를 확인하세요.

In [ ]:
class SR117ModelValidator:
    """SR 11-7 compliant three-stage model validation framework"""

    def __init__(self, model, challenger_models: Dict = None):
        self.model = model
        self.challengers = challenger_models or {}
        self.results: List[ValidationResult] = []

    # ---- Stage 1: Conceptual Soundness ----
    def stage1_conceptual_soundness(self, X_train, y_train, X_test, y_test):
        """Conceptual soundness review: overfitting, feature stability"""
        results = []

        # Overfitting check: train vs test performance gap
        train_pred = self.model.predict_proba(X_train)[:, 1]
        test_pred = self.model.predict_proba(X_test)[:, 1]
        train_auc = roc_auc_score(y_train, train_pred)
        test_auc = roc_auc_score(y_test, test_pred)
        overfit_gap = train_auc - test_auc

        results.append(ValidationResult(
            stage="Conceptual Soundness",
            test_name="Overfitting Check",
            passed=overfit_gap < 0.05,
            score=overfit_gap,
            threshold=0.05,
            details=f"Train AUC: {train_auc:.4f}, Test AUC: {test_auc:.4f}"
        ))

        # Feature importance stability (bootstrap)
        importances = []
        for i in range(100):
            idx = np.random.choice(len(X_train), len(X_train), replace=True)
            boot_model = type(self.model)(**self.model.get_params())
            boot_model.fit(X_train[idx], y_train[idx])
            importances.append(boot_model.feature_importances_)

        cv_importance = np.std(importances, axis=0) / (np.mean(importances, axis=0) + 1e-8)
        max_cv = np.max(cv_importance)

        results.append(ValidationResult(
            stage="Conceptual Soundness",
            test_name="Feature Stability (Bootstrap CV)",
            passed=max_cv < 0.5,
            score=max_cv,
            threshold=0.5,
            details=f"Max feature importance CV: {max_cv:.4f}"
        ))

        self.results.extend(results)
        return results

    # ---- Stage 2: Outcomes Analysis ----
    def stage2_outcomes_analysis(self, X_test, y_test, segments: Dict[str, np.ndarray]):
        """Outcomes analysis: accuracy, calibration, segment performance"""
        results = []

        # Overall discriminatory power
        predictions = self.model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, predictions)
        results.append(ValidationResult(
            stage="Outcomes Analysis",
            test_name="Discriminatory Power (AUC)",
            passed=auc >= 0.70,
            score=auc,
            threshold=0.70,
            details=f"Model AUC: {auc:.4f}"
        ))

        # Calibration (Brier Score)
        brier = brier_score_loss(y_test, predictions)
        results.append(ValidationResult(
            stage="Outcomes Analysis",
            test_name="Calibration (Brier Score)",
            passed=brier < 0.15,
            score=brier,
            threshold=0.15,
            details=f"Brier Score: {brier:.4f}"
        ))

        # Segment-level performance analysis
        for seg_name, seg_mask in segments.items():
            seg_auc = roc_auc_score(y_test[seg_mask], predictions[seg_mask])
            results.append(ValidationResult(
                stage="Outcomes Analysis",
                test_name=f"Segment AUC - {seg_name}",
                passed=seg_auc >= 0.65,
                score=seg_auc,
                threshold=0.65,
                details=f"Segment '{seg_name}' AUC: {seg_auc:.4f}"
            ))

        self.results.extend(results)
        return results

    # ---- Stage 3: Benchmarking ----
    def stage3_benchmarking(self, X_test, y_test):
        """Benchmarking against challenger models"""
        results = []
        champion_pred = self.model.predict_proba(X_test)[:, 1]
        champion_auc = roc_auc_score(y_test, champion_pred)

        for name, challenger in self.challengers.items():
            challenger_pred = challenger.predict_proba(X_test)[:, 1]
            challenger_auc = roc_auc_score(y_test, challenger_pred)
            margin = champion_auc - challenger_auc

            results.append(ValidationResult(
                stage="Benchmarking",
                test_name=f"vs {name}",
                passed=margin >= -0.02,
                score=margin,
                threshold=-0.02,
                details=f"Champion: {champion_auc:.4f}, "
                        f"Challenger({name}): {challenger_auc:.4f}"
            ))

        self.results.extend(results)
        return results

    def generate_validation_report(self) -> Dict:
        """Generate comprehensive validation report (MVR)"""
        total = len(self.results)
        passed = sum(1 for r in self.results if r.passed)

        report = {
            "report_date": datetime.now().isoformat(),
            "overall_status": "APPROVED" if passed == total else "CONDITIONAL" if passed / total > 0.8 else "REJECTED",
            "total_tests": total,
            "passed_tests": passed,
            "failed_tests": [
                {"stage": r.stage, "test": r.test_name, "score": r.score, "threshold": r.threshold}
                for r in self.results if not r.passed
            ],
            "stages": {
                stage: {
                    "passed": sum(1 for r in self.results if r.stage == stage and r.passed),
                    "total": sum(1 for r in self.results if r.stage == stage)
                }
                for stage in ["Conceptual Soundness", "Outcomes Analysis", "Benchmarking"]
            }
        }
        return report

**Financial Data Fabric Gateway: Secure AI Data Access Layer**

In [ ]:
import hashlib
import json
from datetime import datetime
from typing import Dict, List, Optional
from enum import Enum

class DataSensitivity(Enum):
    PUBLIC = "public"
    INTERNAL = "internal"
    CONFIDENTIAL = "confidential"
    RESTRICTED = "restricted"

class AccessDecision(Enum):
    ALLOW = "allow"
    DENY = "deny"
    ALLOW_MASKED = "allow_masked"
    ALLOW_AGGREGATED = "allow_aggregated"

> 아래 셀을 실행하여 결과를 확인하세요.

In [ ]:
class FinancialDataFabricGateway:
    """Data Fabric Gateway for financial network separation compliance"""

    def __init__(self, policy_config: Dict):
        self.policies = policy_config
        self.access_log: List[Dict] = []
        self.data_catalog: Dict = {}

    def register_dataset(self, dataset_id: str, sensitivity: DataSensitivity,
                         columns: List[str], pii_columns: List[str]):
        """Register dataset in the data catalog with metadata"""
        self.data_catalog[dataset_id] = {
            "sensitivity": sensitivity,
            "columns": columns,
            "pii_columns": pii_columns,
            "registered_at": datetime.now().isoformat()
        }

    def request_access(self, requester: str, dataset_id: str,
                       purpose: str, zone: str) -> AccessDecision:
        """Evaluate data access request based on policy"""
        dataset = self.data_catalog.get(dataset_id)
        if not dataset:
            return AccessDecision.DENY

        sensitivity = dataset["sensitivity"]

        # Network zone-based access control
        if zone == "cloud" and sensitivity == DataSensitivity.RESTRICTED:
            self._log_access(requester, dataset_id, AccessDecision.DENY,
                           "Restricted data cannot leave internal network")
            return AccessDecision.DENY

        if zone == "cloud" and sensitivity == DataSensitivity.CONFIDENTIAL:
            self._log_access(requester, dataset_id, AccessDecision.ALLOW_MASKED,
                           "PII columns masked for cloud access")
            return AccessDecision.ALLOW_MASKED

        if zone == "cloud" and sensitivity == DataSensitivity.INTERNAL:
            self._log_access(requester, dataset_id, AccessDecision.ALLOW_AGGREGATED,
                           "Only aggregated statistics allowed")
            return AccessDecision.ALLOW_AGGREGATED

        self._log_access(requester, dataset_id, AccessDecision.ALLOW,
                       f"Access granted in {zone} zone")
        return AccessDecision.ALLOW

    def _log_access(self, requester, dataset_id, decision, reason):
        """Immutable access logging for audit trail"""
        entry = {
            "timestamp": datetime.now().isoformat(),
            "requester": requester,
            "dataset_id": dataset_id,
            "decision": decision.value,
            "reason": reason,
            "hash": hashlib.sha256(
                f"{requester}{dataset_id}{datetime.now()}".encode()
            ).hexdigest()[:16]
        }
        self.access_log.append(entry)

    def generate_audit_report(self) -> Dict:
        """Generate access audit report for FSS inspection"""
        total = len(self.access_log)
        denied = sum(1 for e in self.access_log if e["decision"] == "deny")
        masked = sum(1 for e in self.access_log if e["decision"] == "allow_masked")
        return {
            "report_period": datetime.now().strftime("%Y-%m"),
            "total_requests": total,
            "denied_requests": denied,
            "masked_requests": masked,
            "compliance_rate": (total - denied) / total if total > 0 else 1.0
        }

**PSI-based Model Stability Monitoring**

In [ ]:
def calculate_psi(expected, actual, buckets=10):
 """Calculate PSI (Population Stability Index): retraining needed if >= 0.25"""
 def scale_range(input, min, max):
 return (input - min) / (max - min)

 expected_percents = np.histogram(expected, buckets)[0] / len(expected)
 actual_percents = np.histogram(actual, buckets)[0] / len(actual)

 # Prevent zero variance
 expected_percents = np.clip(expected_percents, 0.0001, 1)
 actual_percents = np.clip(actual_percents, 0.0001, 1)

 psi = np.sum((actual_percents - expected_percents) * np.log(actual_percents / expected_percents))
 return psi

**Financial AI Model Inventory Management System**

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Optional
from datetime import datetime, timedelta
from enum import Enum
import json

class ModelTier(Enum):
    TIER_1 = "Critical"       # Credit scoring, FDS core
    TIER_2 = "Important"      # Risk models, pricing
    TIER_3 = "Standard"       # Marketing, analytics
    TIER_4 = "Low Impact"     # Internal reporting

class ModelStatus(Enum):
    DEVELOPMENT = "development"
    VALIDATION = "validation"
    PRODUCTION = "production"
    MONITORING = "monitoring"
    RETIRED = "retired"

@dataclass
class ModelEntry:
    model_id: str
    model_name: str
    tier: ModelTier
    status: ModelStatus
    business_purpose: str
    algorithm: str
    owner: str
    validator: str
    deploy_date: Optional[datetime] = None
    last_validation: Optional[datetime] = None
    next_validation: Optional[datetime] = None
    psi_current: float = 0.0
    auc_current: float = 0.0
    issues: List[str] = field(default_factory=list)

> 아래 셀을 실행하여 결과를 확인하세요.

In [ ]:
class ModelInventoryManager:
    """Enterprise-wide financial AI model inventory"""

    def __init__(self):
        self.inventory: Dict[str, ModelEntry] = {}
        self.validation_schedule = {
            ModelTier.TIER_1: timedelta(days=90),    # Quarterly
            ModelTier.TIER_2: timedelta(days=180),   # Semi-annually
            ModelTier.TIER_3: timedelta(days=365),   # Annually
            ModelTier.TIER_4: timedelta(days=730),   # Bi-annually
        }

    def register_model(self, entry: ModelEntry):
        """Register a new model in the inventory"""
        entry.next_validation = datetime.now() + self.validation_schedule[entry.tier]
        self.inventory[entry.model_id] = entry

    def get_overdue_validations(self) -> List[ModelEntry]:
        """Identify models with overdue validation"""
        overdue = []
        for model in self.inventory.values():
            if model.status == ModelStatus.PRODUCTION:
                if model.next_validation and model.next_validation < datetime.now():
                    overdue.append(model)
        return sorted(overdue, key=lambda m: m.tier.value)

    def get_high_risk_models(self) -> List[ModelEntry]:
        """Identify models requiring immediate attention"""
        high_risk = []
        for model in self.inventory.values():
            if model.status != ModelStatus.PRODUCTION:
                continue
            risk_flags = []
            if model.psi_current >= 0.25:
                risk_flags.append("PSI exceeds threshold")
            if model.auc_current < 0.65:
                risk_flags.append("AUC below minimum")
            if model.next_validation and model.next_validation < datetime.now():
                risk_flags.append("Validation overdue")
            if risk_flags:
                model.issues = risk_flags
                high_risk.append(model)
        return high_risk

    def generate_board_report(self) -> Dict:
        """Generate model risk summary for board reporting"""
        total = len(self.inventory)
        by_tier = {}
        for tier in ModelTier:
            models = [m for m in self.inventory.values() if m.tier == tier]
            by_tier[tier.value] = {
                "count": len(models),
                "in_production": sum(1 for m in models if m.status == ModelStatus.PRODUCTION),
                "overdue_validation": sum(1 for m in models if m.next_validation and m.next_validation < datetime.now()),
                "high_psi": sum(1 for m in models if m.psi_current >= 0.25)
            }
        return {
            "report_date": datetime.now().isoformat(),
            "total_models": total,
            "models_by_tier": by_tier,
            "high_risk_models": len(self.get_high_risk_models()),
            "overdue_validations": len(self.get_overdue_validations())
        }

**Credit Scoring Fairness Validation with Fairlearn**

In [ ]:
import numpy as np
import pandas as pd
from fairlearn.metrics import (
    MetricFrame, demographic_parity_difference,
    equalized_odds_difference, selection_rate
)
from fairlearn.reductions import ExponentiatedGradient, DemographicParity
from sklearn.metrics import accuracy_score, recall_score, precision_score
from typing import Dict, List, Tuple

class CreditFairnessValidator:
    """Comprehensive fairness validation for credit scoring AI"""

    def __init__(self, protected_attributes: List[str],
                 proxy_correlation_threshold: float = 0.3):
        self.protected_attrs = protected_attributes
        self.proxy_threshold = proxy_correlation_threshold

    def detect_proxy_variables(self, df: pd.DataFrame,
                                feature_columns: List[str]) -> Dict[str, List]:
        """Detect proxy variables correlated with protected attributes"""
        proxy_report = {}
        for attr in self.protected_attrs:
            if attr not in df.columns:
                continue
            proxies = []
            for col in feature_columns:
                if col == attr:
                    continue
                try:
                    corr = abs(df[attr].astype(float).corr(df[col].astype(float)))
                    if corr > self.proxy_threshold:
                        proxies.append({
                            "feature": col,
                            "correlation": round(corr, 4),
                            "action": "REMOVE" if corr > 0.5 else "MONITOR"
                        })
                except (ValueError, TypeError):
                    pass
            proxy_report[attr] = proxies
        return proxy_report

    def compute_fairness_metrics(self, y_true, y_pred,
                                  sensitive_features) -> Dict:
        """Compute comprehensive fairness metrics"""
        metrics = {
            "accuracy": accuracy_score,
            "selection_rate": selection_rate,
            "recall": recall_score,
            "precision": precision_score
        }

        metric_frame = MetricFrame(
            metrics=metrics,
            y_true=y_true,
            y_pred=y_pred,
            sensitive_features=sensitive_features
        )

        dp_diff = demographic_parity_difference(
            y_true, y_pred, sensitive_features=sensitive_features
        )
        eo_diff = equalized_odds_difference(
            y_true, y_pred, sensitive_features=sensitive_features
        )

        # 4/5 rule (Disparate Impact): min_rate / max_rate >= 0.8
        rates = metric_frame.by_group["selection_rate"]
        disparate_impact = rates.min() / rates.max() if rates.max() > 0 else 0

        return {
            "demographic_parity_diff": round(dp_diff, 4),
            "equalized_odds_diff": round(eo_diff, 4),
            "disparate_impact_ratio": round(disparate_impact, 4),
            "four_fifths_rule_pass": disparate_impact >= 0.8,
            "by_group_metrics": metric_frame.by_group.to_dict(),
            "overall_metrics": metric_frame.overall.to_dict()
        }

    def mitigate_bias(self, X_train, y_train, sensitive_features, estimator):
        """Apply bias mitigation using Exponentiated Gradient"""
        mitigator = ExponentiatedGradient(
            estimator=estimator,
            constraints=DemographicParity()
        )
        mitigator.fit(X_train, y_train, sensitive_features=sensitive_features)
        return mitigator

    def generate_fairness_report(self, y_true, y_pred,
                                  sensitive_features, model_name: str) -> Dict:
        """Generate regulatory-compliant fairness report"""
        metrics = self.compute_fairness_metrics(y_true, y_pred, sensitive_features)

        violations = []
        if not metrics["four_fifths_rule_pass"]:
            violations.append("4/5 Rule violation detected")
        if abs(metrics["demographic_parity_diff"]) > 0.1:
            violations.append("Demographic parity threshold exceeded")
        if abs(metrics["equalized_odds_diff"]) > 0.1:
            violations.append("Equalized odds threshold exceeded")

        return {
            "model_name": model_name,
            "report_date": pd.Timestamp.now().isoformat(),
            "protected_attributes": list(sensitive_features.unique()),
            "metrics": metrics,
            "violations": violations,
            "overall_status": "PASS" if not violations else "FAIL",
            "recommendation": "Model approved for production" if not violations
                            else "Bias mitigation required before deployment"
        }

**Cost-sensitive Learning for FDS Recall**

In [ ]:
import torch.nn as nn

class WeightedFDSLoss(nn.Module):
    """
    Weighted loss function for FDS recall enhancement
    """
    def __init__(self, fn_weight=50.0):
        super().__init__()
        self.fn_weight = fn_weight # Penalty for False Negatives

    def forward(self, inputs, targets):
        # Weighting for targets == 1
        probs = nn.functional.sigmoid(inputs)
        loss = -(self.fn_weight * targets * torch.log(probs) + \
                (1 - targets) * torch.log(1 - probs))
        return loss.mean()

**FDS Real-time Detection Architecture: Multi-layer Fraud Scoring Pipeline**

In [ ]:
import time
import numpy as np
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
from datetime import datetime
from enum import Enum
from collections import deque

class FraudDecision(Enum):
    APPROVE = "approve"
    BLOCK = "block"
    REVIEW = "manual_review"
    STEP_UP = "step_up_auth"

@dataclass
class Transaction:
    txn_id: str
    customer_id: str
    amount: float
    merchant_category: str
    location: str
    timestamp: datetime
    channel: str
    device_fingerprint: str

@dataclass
class FDSResult:
    txn_id: str
    decision: FraudDecision
    risk_score: float
    layer_scores: Dict[str, float]
    explanation: List[str]
    processing_time_ms: float

> 아래 셀을 실행하여 결과를 확인하세요.

In [ ]:
class RealTimeFDSPipeline:
    """Multi-layer real-time FDS architecture"""

    def __init__(self, rule_engine, ml_model, gnn_model,
                 block_threshold=0.9, review_threshold=0.7):
        self.rule_engine = rule_engine
        self.ml_model = ml_model
        self.gnn_model = gnn_model
        self.block_threshold = block_threshold
        self.review_threshold = review_threshold
        self.feature_store = {}
        self.customer_history: Dict[str, deque] = {}

    def _update_realtime_features(self, txn: Transaction):
        """Update real-time feature store for customer"""
        cid = txn.customer_id
        if cid not in self.customer_history:
            self.customer_history[cid] = deque(maxlen=100)
        self.customer_history[cid].append(txn)

        history = self.customer_history[cid]
        recent = [t for t in history
                  if (txn.timestamp - t.timestamp).seconds < 3600]

        self.feature_store[cid] = {
            "txn_count_1h": len(recent),
            "total_amount_1h": sum(t.amount for t in recent),
            "unique_merchants_1h": len(set(t.merchant_category for t in recent)),
            "unique_locations_1h": len(set(t.location for t in recent)),
            "max_amount_1h": max((t.amount for t in recent), default=0),
            "amount_velocity": txn.amount / (np.mean([t.amount for t in history]) + 1)
        }

    def process_transaction(self, txn: Transaction) -> FDSResult:
        """Process transaction through multi-layer detection"""
        start = time.time()
        explanations = []
        layer_scores = {}

        # Layer 1: Rule-based instant filter (< 10ms)
        rule_score, rule_reasons = self.rule_engine.evaluate(txn)
        layer_scores["rule_engine"] = rule_score
        if rule_score >= 1.0:  # Hard block rules
            explanations.extend(rule_reasons)
            return FDSResult(
                txn_id=txn.txn_id, decision=FraudDecision.BLOCK,
                risk_score=1.0, layer_scores=layer_scores,
                explanation=explanations,
                processing_time_ms=(time.time() - start) * 1000
            )

        # Layer 2: ML model scoring (< 100ms)
        self._update_realtime_features(txn)
        features = self.feature_store.get(txn.customer_id, {})
        ml_score = self.ml_model.predict_proba(features)
        layer_scores["ml_model"] = ml_score

        # Layer 3: GNN network analysis (< 1000ms, only for suspicious)
        if ml_score > 0.5:
            gnn_score = self.gnn_model.analyze_network(txn.customer_id, txn)
            layer_scores["gnn_analysis"] = gnn_score
        else:
            gnn_score = 0.0
            layer_scores["gnn_analysis"] = gnn_score

        # Ensemble scoring with weighted combination
        final_score = (0.2 * rule_score + 0.5 * ml_score + 0.3 * gnn_score)

        # Decision logic
        if final_score >= self.block_threshold:
            decision = FraudDecision.BLOCK
            explanations.append(f"High risk: combined score {final_score:.3f}")
        elif final_score >= self.review_threshold:
            decision = FraudDecision.REVIEW
            explanations.append(f"Suspicious: manual review needed")
        else:
            decision = FraudDecision.APPROVE

        elapsed = (time.time() - start) * 1000
        return FDSResult(
            txn_id=txn.txn_id, decision=decision,
            risk_score=final_score, layer_scores=layer_scores,
            explanation=explanations, processing_time_ms=elapsed
        )

**Robo-Advisor Suitability Validation Framework**

In [ ]:
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Dict, List, Tuple
from enum import Enum
from datetime import datetime

class RiskProfile(Enum):
    CONSERVATIVE = 1
    MODERATE_CONSERVATIVE = 2
    MODERATE = 3
    MODERATE_AGGRESSIVE = 4
    AGGRESSIVE = 5

@dataclass
class SuitabilityCheck:
    customer_id: str
    profile: RiskProfile
    check_date: datetime
    portfolio_volatility: float
    max_allowed_volatility: float
    churning_ratio: float
    max_drawdown: float
    is_suitable: bool
    violations: List[str]

> 아래 셀을 실행하여 결과를 확인하세요.

In [ ]:
class RoboAdvisorGovernance:
    """Robo-Advisor suitability and governance validation"""

    RISK_LIMITS = {
        RiskProfile.CONSERVATIVE: {"max_vol": 0.05, "max_equity": 0.20, "max_dd": 0.05},
        RiskProfile.MODERATE_CONSERVATIVE: {"max_vol": 0.08, "max_equity": 0.40, "max_dd": 0.08},
        RiskProfile.MODERATE: {"max_vol": 0.12, "max_equity": 0.60, "max_dd": 0.12},
        RiskProfile.MODERATE_AGGRESSIVE: {"max_vol": 0.16, "max_equity": 0.80, "max_dd": 0.18},
        RiskProfile.AGGRESSIVE: {"max_vol": 0.22, "max_equity": 1.00, "max_dd": 0.25},
    }

    def __init__(self, churning_threshold: float = 6.0):
        self.churning_threshold = churning_threshold  # Annual turnover limit
        self.audit_log: List[SuitabilityCheck] = []

    def validate_suitability(self, customer_id: str, profile: RiskProfile,
                              portfolio_returns: np.ndarray,
                              equity_weight: float,
                              annual_turnover: float) -> SuitabilityCheck:
        """Validate portfolio suitability against investor profile"""
        limits = self.RISK_LIMITS[profile]
        violations = []

        # Volatility check
        realized_vol = np.std(portfolio_returns) * np.sqrt(252)
        if realized_vol > limits["max_vol"] * 1.2:  # 20% tolerance
            violations.append(
                f"Volatility {realized_vol:.2%} exceeds limit "
                f"{limits['max_vol']:.2%} by > 20%"
            )

        # Equity weight check
        if equity_weight > limits["max_equity"]:
            violations.append(
                f"Equity weight {equity_weight:.2%} exceeds "
                f"limit {limits['max_equity']:.2%}"
            )

        # Maximum drawdown check
        cumulative = np.cumprod(1 + portfolio_returns)
        running_max = np.maximum.accumulate(cumulative)
        drawdown = (running_max - cumulative) / running_max
        max_dd = np.max(drawdown)
        if max_dd > limits["max_dd"]:
            violations.append(
                f"Max drawdown {max_dd:.2%} exceeds limit {limits['max_dd']:.2%}"
            )

        # Churning check
        if annual_turnover > self.churning_threshold:
            violations.append(
                f"Annual turnover {annual_turnover:.1f}x exceeds "
                f"churning threshold {self.churning_threshold:.1f}x"
            )

        check = SuitabilityCheck(
            customer_id=customer_id,
            profile=profile,
            check_date=datetime.now(),
            portfolio_volatility=realized_vol,
            max_allowed_volatility=limits["max_vol"],
            churning_ratio=annual_turnover,
            max_drawdown=max_dd,
            is_suitable=len(violations) == 0,
            violations=violations
        )

        self.audit_log.append(check)
        return check

    def generate_suitability_report(self) -> Dict:
        """Generate suitability report for regulatory submission"""
        total = len(self.audit_log)
        suitable = sum(1 for c in self.audit_log if c.is_suitable)

        violation_counts = {}
        for check in self.audit_log:
            for v in check.violations:
                category = v.split(" ")[0]
                violation_counts[category] = violation_counts.get(category, 0) + 1

        return {
            "report_date": datetime.now().isoformat(),
            "total_accounts": total,
            "suitable_accounts": suitable,
            "suitability_rate": suitable / total if total > 0 else 1.0,
            "violation_breakdown": violation_counts,
            "status": "COMPLIANT" if suitable / total > 0.95 else "REVIEW_NEEDED"
        }

**Three Lines of Defense Model Implementation for Financial AI**

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Optional
from datetime import datetime
from enum import Enum
import json

class DefenseLine(Enum):
    FIRST = "1st Line (Business)"
    SECOND = "2nd Line (Risk/Compliance)"
    THIRD = "3rd Line (Internal Audit)"

class AlertSeverity(Enum):
    INFO = "info"
    WARNING = "warning"
    CRITICAL = "critical"
    EMERGENCY = "emergency"

@dataclass
class GovernanceAlert:
    alert_id: str
    source_line: DefenseLine
    severity: AlertSeverity
    model_id: str
    description: str
    timestamp: datetime = field(default_factory=datetime.now)
    escalated_to: Optional[DefenseLine] = None
    resolution: Optional[str] = None

class FirstLineDefense:
    """1st Line: Business unit self-governance"""

    def __init__(self, model_id: str):
        self.model_id = model_id
        self.alerts: List[GovernanceAlert] = []

    def daily_monitoring(self, psi: float, auc: float,
                         error_rate: float) -> List[GovernanceAlert]:
        """Daily performance monitoring by business unit"""
        alerts = []

        if psi >= 0.1:
            severity = AlertSeverity.CRITICAL if psi >= 0.25 else AlertSeverity.WARNING
            alerts.append(GovernanceAlert(
                alert_id=f"1L-PSI-{datetime.now().strftime('%Y%m%d')}",
                source_line=DefenseLine.FIRST,
                severity=severity,
                model_id=self.model_id,
                description=f"PSI = {psi:.4f} (threshold: 0.1/0.25)"
            ))

        if auc < 0.70:
            alerts.append(GovernanceAlert(
                alert_id=f"1L-AUC-{datetime.now().strftime('%Y%m%d')}",
                source_line=DefenseLine.FIRST,
                severity=AlertSeverity.CRITICAL,
                model_id=self.model_id,
                description=f"AUC = {auc:.4f} below minimum threshold 0.70"
            ))

        if error_rate > 0.05:
            alerts.append(GovernanceAlert(
                alert_id=f"1L-ERR-{datetime.now().strftime('%Y%m%d')}",
                source_line=DefenseLine.FIRST,
                severity=AlertSeverity.WARNING,
                model_id=self.model_id,
                description=f"Error rate {error_rate:.2%} exceeds 5%"
            ))

        self.alerts.extend(alerts)
        return alerts

class SecondLineDefense:
    """2nd Line: Independent risk management and compliance"""

    def __init__(self):
        self.validation_results: Dict[str, Dict] = {}
        self.policy_violations: List[GovernanceAlert] = []

    def independent_validation(self, model_id: str,
                                validation_result: Dict) -> GovernanceAlert:
        """Perform independent model validation"""
        self.validation_results[model_id] = validation_result

        if validation_result.get("overall_status") == "REJECTED":
            alert = GovernanceAlert(
                alert_id=f"2L-VAL-{model_id}",
                source_line=DefenseLine.SECOND,
                severity=AlertSeverity.EMERGENCY,
                model_id=model_id,
                description="Model validation REJECTED - immediate action required"
            )
            self.policy_violations.append(alert)
            return alert
        return None

    def fairness_audit(self, model_id: str,
                       fairness_report: Dict) -> Optional[GovernanceAlert]:
        """Quarterly fairness audit"""
        if fairness_report.get("overall_status") == "FAIL":
            alert = GovernanceAlert(
                alert_id=f"2L-FAIR-{model_id}",
                source_line=DefenseLine.SECOND,
                severity=AlertSeverity.CRITICAL,
                model_id=model_id,
                description=f"Fairness violation: {fairness_report.get('violations')}"
            )
            self.policy_violations.append(alert)
            return alert
        return None

> 아래 셀을 실행하여 결과를 확인하세요.

In [ ]:
class ThirdLineDefense:
    """3rd Line: Internal audit - independent assurance"""

    def __init__(self):
        self.audit_findings: List[Dict] = []

    def process_audit(self, model_id: str,
                      first_line: FirstLineDefense,
                      second_line: SecondLineDefense) -> Dict:
        """Comprehensive audit of the entire MRM process"""
        findings = []

        # Check if 1st line monitoring is being performed
        if not first_line.alerts:
            findings.append({
                "category": "Monitoring Gap",
                "severity": "HIGH",
                "finding": "No 1st line monitoring alerts found - "
                          "indicates potential monitoring failure"
            })

        # Check if 2nd line validation is current
        validation = second_line.validation_results.get(model_id)
        if not validation:
            findings.append({
                "category": "Validation Gap",
                "severity": "HIGH",
                "finding": "No independent validation record found"
            })

        # Check escalation effectiveness
        critical_alerts = [a for a in first_line.alerts
                          if a.severity in [AlertSeverity.CRITICAL, AlertSeverity.EMERGENCY]]
        unescalated = [a for a in critical_alerts if not a.escalated_to]
        if unescalated:
            findings.append({
                "category": "Escalation Failure",
                "severity": "CRITICAL",
                "finding": f"{len(unescalated)} critical alerts not escalated to 2nd line"
            })

        audit_result = {
            "model_id": model_id,
            "audit_date": datetime.now().isoformat(),
            "findings_count": len(findings),
            "findings": findings,
            "overall_rating": "Satisfactory" if not findings
                            else "Needs Improvement" if len(findings) <= 2
                            else "Unsatisfactory"
        }

        self.audit_findings.append(audit_result)
        return audit_result